In [37]:
from pygom.inference.particle_distance import Distance, EuclidDistance, ScaledDistance, MahalanobisDistance
import numpy as np

# DISTANCE_METHODS = {
#     "euclid": EuclidDistance(),
#     "scaled": ScaledDistance(),
#     "mahalanobis": MahalanobisDistance()
# }

# @property
# def k(self):
#     return self._k

# @k.setter
# def k(self, k):
#     if k < self.n_particles:
#         self._k = k
#     else:
#         raise(ValueError(
#             f"Number of nearest neighbours, {k},"
#             f"must be less than number of particles, {self.n_particles}."
#             ))

class Covariance:
    """
    Calculate covariance matrix (or matrices)
    """

    def prepare(self, distances, particles, weights, tolerance):
        """
        Cache objects we only want to calculate once.
        For now, this is just a was to pre-compute sigma
        """
        pass

    def get_sigma(self, ancestor_idx):
        """
        Get sigma, if no ancestor_idx specified, assumed to be global
        """
        raise NotImplementedError

class GlobalCovariance(Covariance):
    """
    Get covariance matrix of particles
    """
    def prepare(self, distances, particles, weights, tolerance):
        self._sigma = self._compute_global_covariances(particles, tolerance)

    def _compute_global_covariances(self, particles, tolerance):
        return np.cov(particles.T)

    def get_sigma(self, ancestor_idx):
        return self._sigma

class LocalCovariance(Covariance):
    """
    Get local covariance.
    What constitutes "local" may vary
    """
    def __init__(
            self,
            n_particles,
            neighbourhood: str="all",
            k: int|None=None,
            distance_method: Distance|None=None
        ):
        self.n_particles = n_particles
        self.neighbourhood = neighbourhood

        if neighbourhood=="knn":
            self.distance_method = distance_method
            self.k = k

    def prepare(self, distances, particles, weights, tolerance):
        self._sigma_list = self._compute_local_covariances(distances, particles, weights, tolerance)

    def _compute_local_covariances(self, distances, particles, weights, tolerance):
        """
        Find the covariance matrix of the M nearest particles to a specified particle
        """

        if self.neighbourhood=="all":

            # Tolerated subset of particles
            accepted_idx = distances < tolerance
            particles_tol = particles[accepted_idx]
            weights_tol = weights[accepted_idx]
            weights_tol = weights_tol / weights_tol.sum()

            sigma_list = []
            for centre in particles:
                diff = particles_tol - centre
                sigma = np.einsum("ij,ik,i->jk", diff, diff, weights_tol)
                sigma += np.eye(sigma.shape[0]) * 1e-8
                sigma_list.append(sigma)

        elif self.neighbourhood=="knn":

            sigma_list = []
            for idx in range(self.n_particles):
                nearest_neighbour_idx = self.distance_method.nearest_neighbours_idx(particles, idx, self.k)
                sigma = np.cov(particles[nearest_neighbour_idx].T)
                sigma_list.append(sigma)

        return sigma_list

    def get_sigma(self, ancestor_idx):
        return self._sigma_list[ancestor_idx]

In [38]:
rng = np.random.default_rng(seed = 0)

means = np.array([0, 0])
stds = np.array([1, 10])

cov = np.array([
    [stds[0], 0.95],
    [0.95, stds[1]]
])

particles = rng.multivariate_normal(means, cov, size=100)
distances = np.linalg.norm(particles, axis=1)
weights = distances**2
weights = weights / sum(weights)
tolerance = np.quantile(distances, 0.5)

In [39]:
cov = GlobalCovariance()

cov.prepare(distances, particles, weights, tolerance) 
cov.get_sigma(0)

array([[0.98650487, 1.28468954],
       [1.28468954, 9.41926703]])

In [40]:
distance_method = ScaledDistance(lengthscales = stds)

cov = LocalCovariance(
    n_particles=len(particles),
    neighbourhood="all",
    k=10,
    distance_method=distance_method
)

cov.prepare(distances, particles, weights, tolerance) 
cov.get_sigma(1)

array([[0.88161241, 0.63254687],
       [0.63254687, 6.56705281]])